# Chapter 13: Deploying scikit-learn Models in Production

*Summary notebook for **scikit-learn Cookbook, Third Edition** (John Sukup, Packt 2025) — Chapter 13.*

Chapter ini merangkum alur membawa model machine learning dari eksperimen menuju penggunaan yang lebih terkelola. Susunan mengikuti topik utama pada bab buku: deployment, serialization/persistence, scaling, monitoring, model life cycle, dan deployment pipelines.

**Chapter at a glance**

| Recipe | Core idea |
|:--|:--|
| Overview of model deployment | Menyiapkan model dan artefak yang diperlukan aplikasi |
| Serialization and persistence techniques | Menyimpan dan memuat kembali model |
| Scaling models for production | Memproses permintaan dalam batch dan mempertimbangkan throughput |
| Monitoring and updating deployed models | Memantau kualitas prediksi dan perubahan data |
| Managing the model life cycle | Melacak versi model dan dependensi |
| Setting up deployment pipelines | Membuat pemeriksaan sebelum model dirilis |

Contoh dibuat mandiri dengan dataset bawaan atau sintetis, sehingga tidak membutuhkan unduhan data eksternal.

In [1]:
import numpy as np
import sklearn
print("scikit-learn version:", sklearn.__version__)
print("NumPy version:", np.__version__)

scikit-learn version: 1.8.0
NumPy version: 2.3.5


> **Catatan penting.** Model tersimpan bukan berarti model siap produksi. Simpan juga informasi preprocessing, versi pustaka, skema fitur, metrik evaluasi, dan asumsi penggunaan. File `pickle`/`joblib` hanya boleh dimuat dari sumber tepercaya karena proses deserialisasi dapat menjalankan kode.

## 1. Overview of Model Deployment

### Ringkasan
Deployment adalah proses mengintegrasikan model terlatih ke dalam aplikasi atau sistem yang menggunakannya untuk membuat prediksi. Model harus menerima fitur dengan bentuk dan urutan yang sama seperti saat pelatihan. Pipeline membantu menggabungkan preprocessing dan estimator menjadi satu objek.

### Contoh: buat pipeline yang siap disimpan
Kita gunakan dataset diabetes bawaan untuk regresi, lalu pisahkan data train dan test. Contoh ini mendemonstrasikan artefak model; belum membuat layanan web atau endpoint jaringan.

In [2]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

pipeline = Pipeline([
    ("scale", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
pipeline.fit(X_train, y_train)
pred = pipeline.predict(X_test)
print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R²:", round(r2_score(y_test, pred), 3))
print("Prediction shape:", pred.shape)

MAE: 42.81
R²: 0.454
Prediction shape: (89,)


**Cara membaca hasil:** MAE mengukur rata-rata selisih absolut prediksi dan nilai aktual; R² menunjukkan proporsi variasi target yang dijelaskan model pada data uji. Nilai ini adalah baseline, bukan jaminan performa pada data produksi.

## 2. Serialization and Persistence Techniques

### Ringkasan
Serialization menyimpan objek terlatih ke berkas agar dapat dimuat kembali tanpa melatih ulang. `joblib` lazim digunakan untuk objek scikit-learn yang membawa array NumPy. Untuk model yang akan dipindahkan, pastikan versi Python dan pustaka kompatibel, dan simpan metadata eksperimen secara terpisah.

In [3]:
from pathlib import Path
import joblib

artifact_path = Path("diabetes_pipeline.joblib")
joblib.dump(pipeline, artifact_path)
loaded_pipeline = joblib.load(artifact_path)
loaded_pred = loaded_pipeline.predict(X_test)
print("Saved to:", artifact_path.resolve())
print("Predictions identical:", np.allclose(pred, loaded_pred))
print("First 5 predictions:", np.round(loaded_pred[:5], 1))

Saved to: /diabetes_pipeline.joblib
Predictions identical: True
First 5 predictions: [139.9 180.  135.7 292.1 123.2]


**Interpretasi:** prediksi sebelum dan sesudah pemuatan seharusnya sama pada lingkungan yang sama. Dalam deployment nyata, simpan juga `requirements.txt` atau environment lockfile, versi model, tanggal pelatihan, sumber data, dan metrik validasi. Jangan memuat artefak dari sumber yang tidak tepercaya.

## 3. Scaling Models for Production

### Ringkasan
Scaling berarti menjaga sistem tetap mampu menangani volume permintaan dan beban komputasi yang meningkat. Pilihan desain bergantung pada latensi, throughput, ukuran data, serta sumber daya. Untuk banyak aplikasi, prediksi batch dapat mengurangi overhead pemanggilan berulang. Benchmark harus dilakukan pada lingkungan yang mendekati produksi.

In [4]:
import time

# Membandingkan pemanggilan batch dengan prediksi per baris pada sampel uji.
X_batch = X_test
start = time.perf_counter()
batch_pred = loaded_pipeline.predict(X_batch)
batch_seconds = time.perf_counter() - start

start = time.perf_counter()
row_pred = np.array([loaded_pipeline.predict(row.reshape(1, -1))[0]
                     for row in X_test])
row_seconds = time.perf_counter() - start
print(f"Batch prediction: {batch_seconds:.6f} seconds")
print(f"Row-by-row:       {row_seconds:.6f} seconds")
print("Equivalent outputs:", np.allclose(batch_pred, row_pred))

Batch prediction: 0.000369 seconds
Row-by-row:       0.012629 seconds
Equivalent outputs: True


**Interpretasi:** pemanggilan batch sering mengurangi overhead Python, tetapi hasil waktu sangat bergantung pada mesin, ukuran batch, dan model. Jangan menyimpulkan kapasitas produksi dari benchmark kecil ini saja. Ukur latensi persentil, throughput, konsumsi memori, dan perilaku saat beban puncak.

## 4. Monitoring and Updating Deployed Models

### Ringkasan
Monitoring membantu mendeteksi penurunan kualitas, perubahan distribusi input (*data drift*), perubahan hubungan input-target (*concept drift*), dan kegagalan operasional. Jika label aktual baru tersedia belakangan, metrik kualitas dapat dihitung setelah label terkumpul. Ambang peringatan sebaiknya ditetapkan berdasarkan kebutuhan aplikasi.

In [5]:
from sklearn.metrics import mean_absolute_error

# Simulasi: label aktual terbaru tersedia setelah prediksi dibuat.
recent_actual = y_test[:30]
recent_prediction = loaded_pipeline.predict(X_test[:30])
recent_mae = mean_absolute_error(recent_actual, recent_prediction)
print("Recent-window MAE:", round(recent_mae, 2))

# Perubahan sederhana pada statistik input sebagai ilustrasi monitoring drift.
train_mean = X_train.mean(axis=0)
recent_mean = X_test[:30].mean(axis=0)
mean_shift = np.abs(recent_mean - train_mean)
print("Mean absolute feature shifts (first 5):", np.round(mean_shift[:5], 3))

Recent-window MAE: 47.63
Mean absolute feature shifts (first 5): [0.002 0.012 0.005 0.006 0.009]


**Catatan:** pergeseran rata-rata fitur hanyalah pemeriksaan ilustratif, bukan uji drift yang lengkap. Gunakan distribusi, skema, kualitas data, dan metrik yang relevan dengan domain. Jangan otomatis melatih ulang hanya karena satu indikator berubah; investigasi penyebab dan validasi kandidat model terlebih dahulu.

## 5. Managing the Model Life Cycle

### Ringkasan
Siklus hidup model mencakup pelatihan, evaluasi, persetujuan, deployment, monitoring, pembaruan, dan kemungkinan rollback. Setiap rilis sebaiknya dapat ditelusuri ke data, kode, parameter, dan lingkungan yang digunakan. Model baru perlu dibandingkan dengan model aktif pada data validasi yang sama sebelum menggantikannya.

In [6]:
# Contoh metadata minimal yang dapat disimpan bersama artefak.
import json
from datetime import datetime, timezone

metadata = {
    "model_name": "diabetes_ridge_pipeline",
    "model_version": "1.0.0",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "scikit_learn_version": sklearn.__version__,
    "estimator": "Pipeline(StandardScaler, Ridge)",
    "test_mae": float(mean_absolute_error(y_test, pred)),
    "test_r2": float(r2_score(y_test, pred)),
    "feature_count": int(X.shape[1])
}
metadata_path = Path("diabetes_pipeline_metadata.json")
metadata_path.write_text(json.dumps(metadata, indent=2))
print(metadata_path.read_text())

{
  "model_name": "diabetes_ridge_pipeline",
  "model_version": "1.0.0",
  "created_utc": "2026-10-31T07:51:20.982341+00:00",
  "scikit_learn_version": "1.8.0",
  "estimator": "Pipeline(StandardScaler, Ridge)",
  "test_mae": 42.81199941834889,
  "test_r2": 0.45414652070698225,
  "feature_count": 10
}


Metadata ini merupakan contoh dasar. Dalam proyek nyata, catat pula hash atau ID dataset, commit kode, parameter lengkap, skema fitur, pemilik model, persetujuan rilis, dan lokasi artefak. Hindari menyimpan data pribadi atau rahasia di metadata.

## 6. Setting Up Deployment Pipelines

### Ringkasan
Deployment pipeline mengotomatiskan pemeriksaan sebelum artefak dirilis. Contoh pemeriksaan sederhana meliputi: berkas dapat dimuat, prediksi menghasilkan nilai berhingga, bentuk input sesuai, serta metrik validasi memenuhi batas minimum. Pemeriksaan ini tidak menggantikan pengujian integrasi dan keamanan.

In [7]:
# Smoke tests untuk artefak model.
reloaded = joblib.load(artifact_path)
smoke_input = X_test[:3]
smoke_output = reloaded.predict(smoke_input)

assert smoke_output.shape == (3,)
assert np.isfinite(smoke_output).all()
assert mean_absolute_error(y_test, reloaded.predict(X_test)) < 100
print("Smoke tests passed.")
print("Output:", np.round(smoke_output, 2))

Smoke tests passed.
Output: [139.86 179.96 135.72]


**Catatan:** ambang MAE `100` di atas hanya contoh agar pemeriksaan dapat dijalankan; ambang rilis sesungguhnya harus ditentukan dari baseline dan kebutuhan bisnis. Pipeline CI/CD yang lebih lengkap juga menjalankan unit test, pemeriksaan dependensi, validasi data, pengujian beban, dan prosedur rollback.

## 7. Practical Exercises and Chapter Summary

### Ringkasan bab
- Satukan preprocessing dan estimator dalam `Pipeline` untuk mengurangi risiko perbedaan preprocessing saat training dan inference.
- Simpan model dengan alat persistence yang sesuai dan hanya muat artefak tepercaya.
- Ukur throughput, latensi, dan penggunaan sumber daya dengan data yang mewakili kondisi nyata.
- Pantau kualitas prediksi serta perubahan input; lakukan pembaruan melalui proses validasi.
- Lacak versi, metadata, dependensi, dan hasil evaluasi agar model dapat direproduksi dan di-rollback.
- Otomatiskan smoke test dan pemeriksaan rilis.

### Latihan mandiri
1. Simpan pipeline dengan nama versi baru, lalu pastikan prediksinya sama setelah dimuat kembali.
2. Bandingkan latensi prediksi untuk ukuran batch 1, 16, 64, dan seluruh data uji.
3. Simulasikan pergeseran fitur dengan menambahkan nilai konstan ke satu kolom. Buat pemeriksaan yang memberi peringatan ketika pergeseran melewati ambang.
4. Tentukan metrik dan ambang rilis yang masuk akal untuk kasus klasifikasi, lalu buat smoke test yang sesuai.
5. Buat metadata yang mencatat parameter model, versi pustaka, metrik validasi, dan versi artefak.

**Kesimpulan:** deployment yang baik bukan hanya menyimpan model; deployment memerlukan reproducibility, pemeriksaan kualitas, monitoring, serta proses pembaruan yang dapat ditelusuri.